# Geocoding — Forward & Reverse

Demonstrates the `geocoder()` function from the `airgap_geo` library, which
provides both **forward geocoding** (place name → coordinates via Nominatim) and
**reverse geocoding** (coordinates → address via Photon).

## Prerequisites

| Service                       | Default port | Role                                |
| ----------------------------- | ------------ | ----------------------------------- |
| Nominatim (forward geocoding) | 8080         | Place name / postcode → lat/lon     |
| Photon (reverse geocoding)    | 2322         | lat/lon → structured address fields |

Start the services manually (see `docker/README.md`), or run the **Start Services**
cell below.


______________________________________________________________________

## Start Services (optional)

Run the cell below to start the combined stack and wait until the geocoding
services respond. **Skip if the services are already running.**


In [ ]:
import pathlib
import subprocess
import time

import requests as _requests

from airgap_geo.settings import NOMINATIM_URL, PHOTON_API


def _find_repo_root(start: pathlib.Path) -> pathlib.Path:
    """Walk up the directory tree until pyproject.toml is found (repo root marker)."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


_REPO_ROOT = _find_repo_root(pathlib.Path().resolve())
_COMPOSE_FILE = _REPO_ROOT / "docker" / "docker-compose.yml"
_ENV_FILE = _REPO_ROOT / ".env"

_HEALTH_TIMEOUT = 120

_ENDPOINTS = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
}


def start_services(timeout: int = _HEALTH_TIMEOUT) -> None:
    """Start the combined Docker Compose stack and poll until geocoding services respond."""
    print(f"Starting stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "up",
            "-d",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    ready = {name: False for name in _ENDPOINTS}
    deadline = time.monotonic() + timeout
    print(f"\nPolling services (timeout {timeout}s) ...")

    while time.monotonic() < deadline:
        for name, url in _ENDPOINTS.items():
            if ready[name]:
                continue
            try:
                _requests.get(url, timeout=3)
                ready[name] = True
                print(f"  \u2713  {name} is up  ({url})")
            except Exception:
                pass
        if all(ready.values()):
            break
        time.sleep(3)

    still_down = [n for n, ok in ready.items() if not ok]
    if still_down:
        print(f"\n[WARNING] Timed out waiting for: {', '.join(still_down)}")
    else:
        print("\nAll services are up and ready.")


start_services()

## Setup — Imports and Configuration


In [ ]:
import importlib
import pprint

import folium
import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder  # noqa: E402
from airgap_geo.settings import NOMINATIM_URL, PHOTON_API  # noqa: E402

client = httpx.AsyncClient()

print("Configured service endpoints")
print(f"  Nominatim (forward geocoding) : {NOMINATIM_URL}")
print(f"  Photon    (reverse geocoding) : {PHOTON_API}")

## Health Check


In [ ]:
_SERVICES = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
}

rows = []
for name, base_url in _SERVICES.items():
    try:
        r = requests.get(base_url, timeout=5)
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": r.status_code,
                "Reachable": "\u2713",
            }
        )
    except Exception:
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": "ERR",
                "Reachable": "\u2717",
            }
        )

pd.DataFrame(rows).set_index("Service")

______________________________________________________________________

## 1 — Forward Geocoding (Nominatim)

`geocoder(location)` accepts a free-text place name or postcode. When the input is not a
valid `lat,lon` pair it forwards the query to **Nominatim** (`/search`), then enriches the
result with a reverse-geocode call to **Photon** to obtain a structured GeoJSON feature.

The return value always contains a top-level `geo` key: `{"lat": float, "lon": float}`.


In [ ]:
# Forward geocode an address
result_address = await geocoder("10 Downing Street, London", client)
pprint.pprint(result_address)

In [ ]:
# A postcode string also resolves via Nominatim
result_postcode_geocode = await geocoder("EC2V 6DN", client)  # Bank of England
geo = result_postcode_geocode.get("geo", {})
print(f"EC2V 6DN  \u2192  lat={geo.get('lat')}, lon={geo.get('lon')}")

In [ ]:
# Map the result - Nominatim result for 10 Downing Street
geo = result_address.get("geo", {})
lat = geo.get("lat", 51.5034)
lon = geo.get("lon", -0.1276)

props = result_address.get("properties", {})
label = props.get("name") or props.get("street") or "10 Downing Street"

m_fwd = folium.Map(location=[lat, lon], zoom_start=16)
folium.Marker(
    [lat, lon],
    popup=folium.Popup(
        f"<b>{label}</b><br>lat={lat:.5f}, lon={lon:.5f}", max_width=250
    ),
    tooltip="Forward geocoding result",
    icon=folium.Icon(color="blue", icon="home"),
).add_to(m_fwd)
m_fwd

______________________________________________________________________

## 2 — Reverse Geocoding (Photon)

When the input to `geocoder()` is a valid `"lat,lon"` string, the coordinate is sent
directly to **Photon** (`/reverse`) without consulting Nominatim. Photon returns a GeoJSON
feature whose `properties` include name, street, city, and country fields.

Photon is backed by a pre-built Elasticsearch index derived from OpenStreetMap data.


In [ ]:
# Reverse geocode Parliament Square
result_rev = await geocoder("51.5007, -0.1246", client)
pprint.pprint(result_rev)

In [ ]:
# Display key properties
props = result_rev.get("properties", {})
geo_rev = result_rev.get("geo", {})

print(f"Coordinates  : {geo_rev.get('lat')}, {geo_rev.get('lon')}")
print(f"Name         : {props.get('name', '-')}")
print(f"Street       : {props.get('street', '-')}")
print(f"City         : {props.get('city', '-')}")
print(f"Country code : {props.get('countrycode', '-')}")

In [ ]:
# Map the reverse geocoding result
lat_r = geo_rev.get("lat", 51.5007)
lon_r = geo_rev.get("lon", -0.1246)
name_r = props.get("name") or "Parliament Square"

m_rev = folium.Map(location=[lat_r, lon_r], zoom_start=16)
folium.Marker(
    [lat_r, lon_r],
    popup=folium.Popup(
        f"<b>{name_r}</b><br>lat={lat_r:.5f}, lon={lon_r:.5f}", max_width=250
    ),
    tooltip="Reverse geocoding result",
    icon=folium.Icon(color="green", icon="map-marker"),
).add_to(m_rev)
m_rev

______________________________________________________________________

## 3 — Error Handling

The geocoding functions follow a defensive pattern:

- **Connection error / timeout** → `except Exception` swallows and returns `{}`
- **Empty Nominatim results** (nonsense query) → returns `{}`


In [ ]:
from unittest.mock import patch

from airgap_geo.geocoding import photon_reverse_geocode

# Case 1: unreachable service (bad port)
with patch("airgap_geo.geocoding.PHOTON_API", "http://localhost:19999"):
    bad_rev = await photon_reverse_geocode(51.5, -0.1, client)
print(f"Unreachable Photon \u2192 {bad_rev!r}  (expected: {{}})")

# Case 2: empty Nominatim results (nonsense query)
empty_result = await geocoder("xyzzy_this_place_does_not_exist_anywhere_12345", client)
print(f"Unknown place      \u2192 {empty_result!r}  (expected: {{}})")

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:
    """Stop the combined Docker Compose stack. Data volumes are preserved."""
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()